# Training Loop and Loss Construction: Token-Level or Sentence-Level?

> **Previous recap**: We already have a MiniGPT model. It takes token IDs and outputs logits. The parameters are still random, so now we need to train.
>
> **Core questions**:
> 1. How do we organize training data?
> 2. What is the loss actually computing?
> 3. Is training token-level, or sentence-level?
>
> In this part we use a tiny example and show how every number is computed.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

### 1. Start from a super simple example

Assume we have a tiny model: vocab size 5, and we want to train it to predict the next token.

```
Vocab: [BOS=0, I=1, love=2, you=3, EOS=4]

Only one training sentence: "I love you"
Token IDs: [BOS, I, love, you, EOS]
         = [0,   1, 2,    3,   4]
```

**Training objective**: given the previous tokens, predict the next one.

```
given [BOS]                  -> predict I
given [BOS, I]               -> predict love
given [BOS, I, love]         -> predict you
given [BOS, I, love, you]    -> predict EOS
```

This looks like 4 independent prediction tasks, but Transformers have a trick: **they can predict all positions in parallel.**


### 2. Inputs and labels: one forward pass gives loss for all positions

This is the most important mental model. Look at this picture:

```
Training sentence: [BOS, I, love, you, EOS]
                  [0,   1, 2,    3,   4]

          +---------------------------------+
Input:     |  BOS  |   I   | love  |  you   |
          |  [0]  |  [1]  | [2]   | [3]    |
          +---------------------------------+
                    | model forward
          +---------------------------------+
Model out: | logits | logits| logits| logits|
          |  [0]   |  [1]  |  [2]  |  [3]  |
          +---------------------------------+
                    | each position predicts the next token
          +---------------------------------+
Targets:   |   I   | love  |  you  |  EOS  |
          |  [1]  | [2]   | [3]   | [4]   |
          +---------------------------------+

input  = drop the last token
labels = drop the first token (shift by 1)
```

This is called **teacher forcing**: we do not continue with the model's own predictions, we continue with the correct answers.

Why? Because then all positions can be trained **in parallel**, without waiting for earlier positions.


In [ ]:
# 用手算demo这个概念
sentence = torch.tensor([0, 1, 2, 3, 4])  # [BOS, I, love, you, EOS]

print("Full sentence:", sentence.tolist())
print()

# Input：去掉最后一个
input_ids = sentence[:-1]  # [0, 1, 2, 3]
print("Input (去掉最后一个):", input_ids.tolist())
print("Meaning:                      [BOS,  I,   love,   you ]")
print()

# Labels：去掉第一个（右移一位）
target_ids = sentence[1:]   # [1, 2, 3, 4]
print("Labels (去掉第一个):    ", target_ids.tolist())
print("Meaning:                      [I,   love,   you,   EOS]")
print()

print("One-to-one mapping:")
for i in range(len(input_ids)):
    print(f"  pos {i}: sees [{', '.join(str(x) for x in input_ids[:i+1].tolist())}] -> predict {target_ids[i].item()}")

### 3. How is loss computed? Cross-Entropy Loss

At each position the model outputs logits (one score per vocab item). We compare them against the labels.

We use **cross-entropy loss**:

```
For each position:
1. Convert logits to probabilities: softmax(logits)
2. Take the probability of the correct label
3. loss = -log(prob_correct)
4. Average over positions
```

**Intuition**:
- prob_correct = 1.0 -> loss = 0 (perfect)
- prob_correct = 0.01 -> loss = 4.6 (terrible)

Next we compute it step by step in code.


In [ ]:
# 模拟model的输出
vocab_size = 5
seq_len = 4  # Input长度

# 假设model输出的 logits (batch=1)
# 实际中这些是model算出来的，这里I们手动造一些来观察
torch.manual_seed(123)
logits = torch.randn(1, seq_len, vocab_size)  # [batch=1, seq_len=4, vocab=5]
targets = torch.tensor([[1, 2, 3, 4]])          # [batch=1, seq_len=4]

print(f"model输出 logits 形状: {logits.shape}")
print(f"Labels形状: {targets.shape}")
print()

# 看第 0 个pos的 logits 和Labels
print(f"pos 0 的 logits: {logits[0, 0].tolist()}")
print(f"pos 0 的Labels:   {targets[0, 0].item()}")
print(f"-> model要在 5 个词里猜对答案是词 {targets[0, 0].item()}")

In [ ]:
# 手工算一遍 loss（理解每个数字怎么来的）
print("=== 手工计算 Cross-Entropy Loss ===")
print()

total_loss = 0.0
for pos in range(seq_len):
    # 这个pos的 logits (vocab_size 个分数)
    pos_logits = logits[0, pos]  # [vocab_size]
    # 这个pos的正确答案
    correct_id = targets[0, pos].item()
    
    # Step 1: softmax 转概率
    probs = F.softmax(pos_logits, dim=-1)
    
    # Step 2: 正确答案的概率
    correct_prob = probs[correct_id].item()
    
    # Step 3: loss = -log(概率)
    pos_loss = -math.log(correct_prob)
    total_loss += pos_loss
    
    print(f"pos {pos}: 正确答案=词{correct_id}, 概率={correct_prob:.4f}, loss={pos_loss:.4f}")

# Step 4: 平均
manual_loss = total_loss / seq_len
print(f"\n所有pos平均 loss: {manual_loss:.4f}")

# 对比 PyTorch 内置的 cross_entropy
pt_loss = F.cross_entropy(
    logits.reshape(-1, vocab_size),  # [batch*seq_len, vocab]
    targets.reshape(-1)               # [batch*seq_len]
).item()
print(f"PyTorch cross_entropy: {pt_loss:.4f}")
print(f"两者一致？ {'[ok]' if abs(manual_loss - pt_loss) < 1e-4 else '[x]'}")

### 4. Token-level training or sentence-level training?

**Answer: token-level training.**

But note: it's token-level **and parallel**, not token-by-token serial training.

```
+----------------------------------------------+
|              one forward + backward           |
|                                              |
|  Loss = loss(pos0) + loss(pos1) + ...         |
|                                              |
|  pos 0 predicts token 1                       |
|  pos 1 predicts token 2    } all computed     |
|  pos 2 predicts token 3    } in parallel      |
|  pos 3 predicts token 4                       |
|                                              |
|  gradients sum over all positions             |
|  update parameters <- learning signals from   |
|                      all positions            |
+----------------------------------------------+
```

**Why not sentence-level?**
- Sentence-level means supervising only one position (e.g. at the end) -> signal is too sparse.
- For a 100-token sentence, you'd get only one supervision signal.
- Token-level gives ~100 supervision signals.

**But it is also NOT serial token training.** It is parallel: one forward pass produces all losses.
That is one reason Transformers train much faster than RNNs.


### 5. Training multiple sentences in a batch

Real training does not train on only one sentence. We train batches (e.g. 32 sequences) as a matrix.

```
Batch input:
[[BOS,  I,    love, you,  EOS,  PAD,  PAD],   <- sentence 1 (5 valid tokens)
 [BOS,  hello,world,EOS,  PAD,  PAD,  PAD]]   <- sentence 2 (4 valid tokens)

Shape: [batch_size=2, seq_len=7]
```

Loss: average over all positions across all sentences (excluding PAD positions).

```python
loss = cross_entropy(logits.reshape(-1, vocab), targets.reshape(-1), ignore_index=PAD_ID)
#                                                         ^ ignore padding
```


In [ ]:
# demo batch train的 loss 计算
PAD_ID = 0  # 假设 0 是 PAD

batch_input = torch.tensor([
    [0, 1, 2, 3, 4, 0, 0],  # [BOS, I, love, you, EOS, PAD, PAD]
    [0, 2, 4, 0, 0, 0, 0],  # [BOS, love, EOS, PAD, PAD, PAD, PAD]
])

# Labels = input 右移一位
batch_target = torch.tensor([
    [1, 2, 3, 4, 0, 0, 0],  # [I, love, you, EOS, PAD, PAD, PAD]
    [2, 4, 0, 0, 0, 0, 0],  # [love, EOS, PAD, PAD, PAD, PAD, PAD]
])

print("Batch Input:")
print(batch_input)
print()
print("Batch Labels:")
print(batch_target)
print()

# 模拟model输出
batch_logits = torch.randn(2, 7, 5)  # [batch=2, seq=7, vocab=5]

# Key：ignore_index=PAD_ID，PAD pos不参与 loss 计算
loss_with_ignore = F.cross_entropy(
    batch_logits.reshape(-1, 5),       # [14, 5]
    batch_target.reshape(-1),           # [14]
    ignore_index=PAD_ID
)

loss_without_ignore = F.cross_entropy(
    batch_logits.reshape(-1, 5),
    batch_target.reshape(-1)
)

print(f"ignore PAD 的 loss: {loss_with_ignore.item():.4f}")
print(f"不ignore PAD 的 loss: {loss_without_ignore.item():.4f}")
print(f"\n差别很大！因为 PAD pos的predict没有意义，不应该贡献 loss。")

### 6. The complete training loop

Now we combine everything.
Because we have not trained a real tokenizer yet, we use a **toy setting** to demonstrate the full training loop.


In [ ]:
# 复用 Part 4 的 MiniGPT，简化版
def get_sinusoidal_encoding(seq_len, d_model):
    position = torch.arange(seq_len).unsqueeze(1)
    div_term = torch.exp(
        torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
    )
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model=64, num_heads=4, num_layers=4, max_seq_len=128):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len
        
        self.token_emb = nn.Embedding(vocab_size, d_model)
        pe = get_sinusoidal_encoding(max_seq_len, d_model)
        self.register_buffer('pe', pe)
        
        # 简化：不用 ModuleList，直接写几个 block
        self.attn1 = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.ffn1 = nn.Sequential(
            nn.Linear(d_model, 4*d_model), nn.ReLU(), nn.Linear(4*d_model, d_model)
        )
        self.norm1a = nn.LayerNorm(d_model)
        self.norm1f = nn.LayerNorm(d_model)
        
        self.attn2 = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.ffn2 = nn.Sequential(
            nn.Linear(d_model, 4*d_model), nn.ReLU(), nn.Linear(4*d_model, d_model)
        )
        self.norm2a = nn.LayerNorm(d_model)
        self.norm2f = nn.LayerNorm(d_model)
        
        self.ln_final = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        batch_size, seq_len = x.shape
        x = self.token_emb(x) + self.pe[:seq_len, :]
        
        mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device) * float('-inf'), diagonal=1)
        
        # Block 1
        attn_out, _ = self.attn1(x, x, x, attn_mask=mask)
        x = self.norm1a(x + attn_out)
        x = self.norm1f(x + self.ffn1(x))
        
        # Block 2
        attn_out, _ = self.attn2(x, x, x, attn_mask=mask)
        x = self.norm2a(x + attn_out)
        x = self.norm2f(x + self.ffn2(x))
        
        x = self.ln_final(x)
        return self.lm_head(x)

print("MiniGPT model定义完成！")

In [ ]:
# === 完整的train循环demo ===

# 1. 准备假数据（模拟 token 化后的文本）
VOCAB_SIZE = 20
PAD_ID = 0
SEQ_LEN = 16
BATCH_SIZE = 8

# 假数据：随机 token 序列
train_data = torch.randint(1, VOCAB_SIZE, (100, SEQ_LEN))  # 100 条「句子」
print(f"train数据: {train_data.shape} (100 条, 每条 {SEQ_LEN} 个 token)")

# 2. 创建model
model = MiniGPT(VOCAB_SIZE, d_model=64, num_heads=4, num_layers=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
print(f"model参数量: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# 3. train循环
NUM_EPOCHS = 5
losses = []

model.train()
for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0
    num_batches = 0
    
    for i in range(0, len(train_data), BATCH_SIZE):
        batch = train_data[i:i+BATCH_SIZE]  # [batch_size, seq_len]
        
        # 准备Input和Labels
        input_ids = batch[:, :-1]      # 去掉最后一个
        target_ids = batch[:, 1:]       # 去掉第一个
        
        # Forward
        logits = model(input_ids)  # [batch, seq_len-1, vocab_size]
        
        # Loss: 把所有 batch 和所有pos展平
        loss = F.cross_entropy(
            logits.reshape(-1, VOCAB_SIZE),  # [batch*(seq_len-1), vocab_size]
            target_ids.reshape(-1)            # [batch*(seq_len-1)]
        )
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        num_batches += 1
    
    avg_loss = epoch_loss / num_batches
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Loss: {avg_loss:.4f}")

print(f"\nLoss 从 {losses[0]:.4f} 降到 {losses[-1]:.4f} -> model在学习！")

In [ ]:
# 可视化 loss 下降
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(losses, 'o-', markersize=8)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training loss curve')
plt.grid(True, alpha=0.3)
plt.show()

print("Loss 在下降 = model在学会predict下一个 token")


### 7. Core recap: one diagram to settle it

```
+--------------------------------------------------------+
|     What does token-level training really mean?         |
+--------------------------------------------------------+
|                                                        |
|  full seq:   [BOS, I, love, you, C, h, EOS]             |
|              | drop the last token                      |
|  model in:   [BOS, I, love, you, C, h]                  |
|              | forward                                  |
|  model out:  [logits_0, logits_1, ..., logits_5]        |
|              | compare with labels                      |
|  labels:     [I, love, you, C, h, EOS]                  |
|                                                        |
|  Loss = mean(loss_0, loss_1, ..., loss_5)               |
|              | backward                                 |
|  gradients from all positions -> update parameters      |
|                                                        |
|  [ok] token-level: each position contributes loss       |
|  [ok] parallel: one forward predicts all positions      |
|  [x] not sentence-level: not only the last position     |
|  [x] not serial tokens: no waiting token by token       |
|                                                        |
+--------------------------------------------------------+
```


### 8. A common confusion: training vs inference

| | training | inference / generation |
|:---|:---|:---|
| Input | full sentence (drop last token) | only a prompt |
| Computation | parallel over positions | token-by-token serial |
| Labels | ground truth (teacher forcing) | previous token generated by the model |
| Mask | hide the future | hide the future |
| Loss | computed | not computed |

**Key difference**:
- training uses teacher forcing -> parallel -> fast
- inference has no answers -> must generate serially -> slow

-> Next part: autoregressive generation.


### 9. Gradient view: what happens after computing loss?

Loss is just one number.
What actually drives learning is the **gradient**.

```
loss (one number)
    | backward
grads (one number per parameter)
    | optimizer.step
params update (model becomes slightly better)
```

This section focuses on gradients, the step people often skip.


#### 9.1 Backpropagation: how gradients flow from loss back to parameters

**Intuition**: loss is the result of all parameters together.
Backprop asks:

> If I increase a parameter by a tiny amount, how much does loss change?

That rate of change is the gradient.

A simplest example (one neuron):

```
input x -> [weight w] -> y = w * x
                     |
                  loss = (y - target)^2
```

**Chain rule**:

```
\partial loss / \partial w = (\partial loss / \partial y) * (\partial y / \partial w)
                           = 2 (y - target) * x
```

In LLMs the chain passes through many Transformer blocks, but the principle is identical.


In [ ]:
# 手动demo：一个简单网络的反向传播
print("=== 手动反向传播demo ===")
print()

# 构建一个最简单的「model」
w = torch.tensor([0.5], requires_grad=True)
b = torch.tensor([0.1], requires_grad=True)

x = torch.tensor([2.0])   # Input
target = torch.tensor([3.0])  # 目标

print(f"参数: w={w.item():.2f}, b={b.item():.2f}")
print(f"Input x={x.item():.2f}, 目标 target={target.item():.2f}")
print()

# Forward
y = w * x + b              # y = 0.5*2 + 0.1 = 1.1
loss = (y - target) ** 2   # loss = (1.1 - 3)^2 = 3.61

print(f"Forward: y = w*x + b = {w.item()}*{x.item()} + {b.item()} = {y.item():.2f}")
print(f"Loss = (y - target)² = ({y.item():.2f} - {target.item():.2f})² = {loss.item():.2f}")
print()

# Backward
loss.backward()

print(f"∂loss/∂w = {w.grad.item():.4f}  (Meaning: w 增加 1, loss 变化 {w.grad.item():.4f})")
print(f"∂loss/∂b = {b.grad.item():.4f}  (Meaning: b 增加 1, loss 变化 {b.grad.item():.4f})")
print()

# 手工验证链式法则
print("=== 手工验证链式法则 ===")
print(f"∂loss/∂y = 2*(y - target) = 2*({y.item():.2f} - {target.item():.2f}) = {2*(y.item()-target.item()):.2f}")
print(f"∂y/∂w = x = {x.item():.2f}")
print(f"∂y/∂b = 1")
print(f"∂loss/∂w = ∂loss/∂y * ∂y/∂w = {2*(y.item()-target.item()):.2f} * {x.item():.2f} = {2*(y.item()-target.item())*x.item():.2f}")
print(f"PyTorch 算的 ∂loss/∂w = {w.grad.item():.4f}  [ok] 一致！")

In [ ]:
# 在 MiniGPT 上看gradient流动
VOCAB_SIZE = 20
model = MiniGPT(VOCAB_SIZE, d_model=64, num_heads=4, num_layers=2)

dummy_input = torch.randint(1, VOCAB_SIZE, (2, 16))   # [batch=2, seq=16]
dummy_target = torch.randint(1, VOCAB_SIZE, (2, 15))  # [batch=2, seq=15]

# Forward
logits = model(dummy_input[:, :-1])
loss = F.cross_entropy(
    logits.reshape(-1, VOCAB_SIZE),
    dummy_target.reshape(-1)
)

# Backward
model.zero_grad()
loss.backward()

print("=== MiniGPT 各层gradientnorm ===")
print()
print(f"{'层':<40s} {'gradientnorm':>12s} {'参数形状':>18s}")
print("-" * 72)
total_grad_norm = 0
for name, param in model.named_parameters():
    if param.grad is not None:
        grad_norm = param.grad.norm().item()
        total_grad_norm += grad_norm ** 2
        param_shape = str(list(param.shape))
        print(f"{name:<40s} {grad_norm:>12.6f}  {param_shape:>18s}")

total_grad_norm = total_grad_norm ** 0.5
print("-" * 72)
print(f"{'总gradientnorm (L2)':<40s} {total_grad_norm:>12.6f}")
print()
print("观察:")
print("  1. 每个参数都有gradient = 反向传播成功把 loss 信号传到了所有层")
print("  2. lm_head（输出层）的gradient通常较大 = loss 信号最强的地方")
print("  3. Embedding 层gradient较小 = 信号经过多层衰减")
print("  4. 残差连接的存在保证gradient不会消失（这是 Transformer 好训的Key！）")

#### 9.2 Token-level gradients: not all tokens are equally important

We said each token contributes to loss. But do they contribute equally?

No.

```
"The capital of France is Paris"
        easy            medium        key info
     (small loss)                  (large loss)
```

- High-frequency tokens like "is" become easy -> small loss -> small gradients
- Content tokens like "Paris" -> larger loss -> larger gradients

So training signal is mostly driven by the hard/content tokens.

This connects to a key RL training issue (e.g. Shuffle-R1): advantage collapsing.
If most rollouts have advantage close to 0, gradient signals become weak.


In [ ]:
# demo：不同 token pos的 loss 不同，产生的gradient也不同
print("=== Token 级 Loss 分析 ===")
print()

VOCAB_SIZE = 20
model = MiniGPT(VOCAB_SIZE, d_model=64, num_heads=4, num_layers=2)

# 构造一个句子：前半是简单 pattern，后半是随机
# 简单 pattern: [1,2,3,1,2,3,1,2,3] 循环
easy_part = torch.tensor([1, 2, 3, 1, 2, 3, 1, 2, 3])
# 随机部分
hard_part = torch.randint(10, VOCAB_SIZE, (7,))
sentence = torch.cat([easy_part, hard_part])  # seq_len=16
batch = sentence.unsqueeze(0)  # [1, 16]

input_ids = batch[:, :-1]      # [1, 15]
target_ids = batch[:, 1:]       # [1, 15]

print(f"句子前半（简单pattern）: {easy_part.tolist()}")
print(f"句子后半（随机token）:   {hard_part.tolist()}")
print()

# Forward - 需要 retain_graph 来逐个pos算
logits = model(input_ids)  # [1, 15, vocab_size]

# 算每个pos的 loss
print("每个 token pos的 loss:")
print(f"{'Position':>4s} {'token':>6s} {'loss':>10s} {'区域'}")
print("-" * 40)

for pos in range(15):
    pos_logits = logits[0, pos]  # [vocab_size]
    pos_target = target_ids[0, pos]
    pos_loss = F.cross_entropy(pos_logits.unsqueeze(0), pos_target.unsqueeze(0)).item()
    region = "简单区" if pos < 8 else "困难区"
    print(f"{pos:>4d} {pos_target.item():>6d} {pos_loss:>10.4f}  {region}")

# 对比简单区和困难区的平均 loss
logits_flat = logits.reshape(-1, VOCAB_SIZE)
targets_flat = target_ids.reshape(-1)
all_losses = F.cross_entropy(logits_flat, targets_flat, reduction='none')

easy_avg = all_losses[:9].mean().item()
hard_avg = all_losses[9:].mean().item()

print()
print(f"简单区平均 loss: {easy_avg:.4f}")
print(f"困难区平均 loss: {hard_avg:.4f}")
print(f"困难区 loss 是简单区的 {hard_avg/easy_avg:.2f} 倍")
print()
print("-> 困难 token 产生更大的gradient，驱动更多学习。")
print("-> 这就是为什么 RL train中需要关注哪些 rollout/token 真正贡献了gradient。")


In [ ]:
# 进阶：可视化 LLM 中 token 级gradient的分布
print("=== Token 级gradient贡献模拟 ===")
print()

# 模拟一句话中每个 token 的gradientnorm
torch.manual_seed(42)

# 模拟 20 个 token 的 loss（前面小，后面大）
token_losses = torch.tensor([0.1, 0.15, 0.2, 0.15, 0.1,
                              0.3, 0.5, 0.8, 1.2, 1.5,
                              2.0, 2.5, 2.8, 3.0, 3.2,
                              3.5, 3.8, 4.0, 4.2, 4.5])

tokens = ["BOS", "I", "是", "一个", "AI",
          "今天", "天气", "真的", "非常", "不错",
          "量子", "纠缠", "是非", "定域", "性的",
          "物理", "现象", "之一", ",", "EOS"]

# gradient ≈ loss（简化：假设gradient正比于 loss）
grad_contrib = token_losses / token_losses.sum() * 100

print("Token 级gradient贡献:")
print(f"{'Token':<8s} {'Loss':>8s} {'gradient贡献':>12s} {'可视化'}")
print("-" * 60)

threshold = 5.0  # 超过 5% 算「高贡献」
high_count = 0
for i in range(len(tokens)):
    bar_len = int(grad_contrib[i].item() * 3)
    bar = "█" * bar_len
    marker = " ★" if grad_contrib[i] > threshold else ""
    if grad_contrib[i] > threshold:
        high_count += 1
    print(f"{tokens[i]:<8s} {token_losses[i].item():>8.2f} {grad_contrib[i].item():>10.1f}% {bar}{marker}")

print()
print(f"gradient贡献 > {threshold}% 的 token: {high_count}/{len(tokens)} 个")
print(f"这 {high_count} 个 token 贡献了 {grad_contrib[-high_count:].sum().item():.1f}% 的gradient")
print()
print("Key洞察:")
print("  1. 前几个 token（高频词）几乎不贡献gradient")
print("  2. 后几个 token（内容词、难词）贡献了绝大部分gradient")
print("  3. 这意味着train效率由「最难的几个 token」决定")
print("  4. Shuffle-R1 的 PTS + ABS 正是为了解决 RL 中这个问题")

#### 9.3 Gradient clipping: prevent gradient explosion

**Problem**: sometimes a batch produces very large gradients.
If the optimizer takes a huge step, loss can become NaN and training collapses.

**Solution**: gradient clipping.

```
if ||grad||_2 > max_norm:
    scale gradients down to make the norm = max_norm
else:
    keep unchanged
```

In LLM training clipping is very common, often `max_norm=1.0`.


In [ ]:
# demogradient裁剪
print("=== gradient裁剪demo ===")
print()

# 创建一个小网络，手动制造一个爆炸的gradient
linear = nn.Linear(10, 1)

x = torch.randn(5, 10)
target = torch.randn(5, 1) * 100  # 故意放大 target，制造大gradient

# 不裁剪
loss = F.mse_loss(linear(x), target)
loss.backward()

raw_grad_norm = sum(p.grad.norm().item() ** 2 for p in linear.parameters()) ** 0.5
print(f"不裁剪的gradientnorm: {raw_grad_norm:.4f}")

# 重置
linear.zero_grad()

# 裁剪
loss = F.mse_loss(linear(x), target)
loss.backward()
max_norm = 1.0
nn.utils.clip_grad_norm_(linear.parameters(), max_norm)
clipped_grad_norm = sum(p.grad.norm().item() ** 2 for p in linear.parameters()) ** 0.5

print(f"裁剪后的gradientnorm: {clipped_grad_norm:.4f} (上限={max_norm})")
print()
print(f"原始gradient太大 -> clip 到 {max_norm} -> train不会崩")
print()

# 实战：train循环中加入gradient裁剪
print("实战代码片段:")
print("```python")
print("loss.backward()")
print("torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)")
print("optimizer.step()")
print("```")
print()
print("这行 clip 是youtrain LLM 时的「安全带」——")
print("平时看不出作用，但Key时刻拯救you的train。")

#### 9.4 Gradient accumulation: simulate a large batch on a small GPU

**Problem**: training wants big batch (e.g. 512), but your GPU fits batch=4.

Key observation:

```
Grad(batch=8) = Grad(batch=4) + Grad(batch=4)
```

So you can run many micro-batches, accumulate gradients, and call `optimizer.step()` once.


In [ ]:
# demogradient累积
print("=== gradient累积demo ===")
print()

VOCAB_SIZE = 20
model_small = MiniGPT(VOCAB_SIZE, d_model=32, num_heads=2, num_layers=1)
model_large = MiniGPT(VOCAB_SIZE, d_model=32, num_heads=2, num_layers=1)

# 复制相同参数
model_large.load_state_dict(model_small.state_dict())

# 假数据
all_data = torch.randint(1, VOCAB_SIZE, (16, 16))  # 总共 16 条

# --- 方式 A: 大 batch (batch=16) 直接训 ---
opt_large = torch.optim.SGD(model_large.parameters(), lr=0.01)

input_large = all_data[:, :-1]
target_large = all_data[:, 1:]
logits_large = model_large(input_large)
loss_large = F.cross_entropy(
    logits_large.reshape(-1, VOCAB_SIZE),
    target_large.reshape(-1)
)
opt_large.zero_grad()
loss_large.backward()

# 保存大 batch 的gradient
grads_large = {name: p.grad.clone() for name, p in model_large.named_parameters() if p.grad is not None}
opt_large.step()

# --- 方式 B: 小 batch (batch=4) + gradient累积 4 次 ---
opt_small = torch.optim.SGD(model_small.parameters(), lr=0.01)
opt_small.zero_grad()

ACCUM_STEPS = 4
small_batch_size = 4
for step in range(ACCUM_STEPS):
    start = step * small_batch_size
    end = start + small_batch_size
    mini_batch = all_data[start:end]

    input_small = mini_batch[:, :-1]
    target_small = mini_batch[:, 1:]
    logits_small = model_small(input_small)
    loss_small = F.cross_entropy(
        logits_small.reshape(-1, VOCAB_SIZE),
        target_small.reshape(-1)
    )

    # Key：loss 除以累积步数，保持gradient量级一致
    (loss_small / ACCUM_STEPS).backward()
    print(f"  累积步 {step+1}/{ACCUM_STEPS}: loss={loss_small.item():.4f}, gradient已累加（不更新）")

print()

# 保存小 batch 累积的gradient
grads_small = {name: p.grad.clone() for name, p in model_small.named_parameters() if p.grad is not None}
opt_small.step()

# 对比两种方式的gradient是否一致
print("=== gradient对比 ===")
all_close = True
for name in grads_large:
    diff = (grads_large[name] - grads_small[name]).norm().item()
    status = "[ok]" if diff < 1e-4 else "[x]"
    if diff >= 1e-4:
        all_close = False
    print(f"  {name:<30s} 差异={diff:.8f} {status}")

print()
if all_close:
    print("Conclusion: gradient累积的gradient == 大 batch 的gradient [ok]")
    print("-> 用 4 次 batch=4 的 forward，模拟了 batch=16 的效果")
else:
    print("注意: 由于顺序处理 mini-batch 时 BatchNorm 行为不同，可能有微小差异")
    print("但对 LLM（只用 LayerNorm）来说是完全等价的")

#### 9.5 Gradient flow panorama (LLM)

Connect everything:

- forward: data -> embedding -> Transformer blocks -> LM head
- loss: cross-entropy per token
- backward: gradients flow back through layers
- gradient handling: clipping, accumulation, etc.
- optimizer.step(): parameter update

Key ideas:
1. residual connections act like gradient highways
2. token-level gradients are uneven
3. RL makes gradient sparsity worse


In [ ]:
# 可视化：残差连接如何保护gradient流动
print("=== 残差连接与gradient流动 ===")
print()

print("没有残差连接（像传统网络）：")
print("  Input -> Layer1 -> Layer2 -> ... -> Layer32 -> Output")
print("  gradient: Output -> Layer32->...-> Layer1 -> Input")
print("  问题: 经过 32 层乘积，gradient可能衰减到 0（gradient消失）")
print()

print("有残差连接（Transformer 做法）：")
print("  Input -> [Layer1 + Input] -> [Layer2 + prev] -> ... -> Output")
print("  gradient: 两条路——")
print("    主路: Output -> Layer32 -> ... -> Layer1 -> Input  (可能衰减)")
print("    短路: Output -> Input  (直接跳过所有层！)  ← gradient高速公路")
print()
print("因为短路的存在，至少有一部分gradient能无损到达底层。")
print("这就是为什么 Transformer 可以堆 100+ 层还能训。")
print()

# 模拟gradient在不同深度下的衰减
depths = [1, 4, 8, 16, 32, 64]

print("模拟：初始gradient=1.0，经过不同层数后的衰减")
print(f"{'层数':>6s}  {'无残差':>12s}  {'有残差':>12s}")
print("-" * 32)

decay_per_layer = 0.95  # 每层衰减 5%
skip_ratio = 0.3         # 残差路径占 30%

for d in depths:
    no_skip = decay_per_layer ** d
    with_skip = no_skip * (1 - skip_ratio) + skip_ratio
    print(f"{d:>6d}  {no_skip:>12.6f}  {with_skip:>12.6f}")

print()
print("Conclusion: 32 层后，无残差gradient只剩 20%，有残差还能保持 44%")
print("层数越多，残差连接的价值越大。")

#### 9.6 Gradient summary

| Concept | One line | Why it matters |
|:---|:---|:---|
| Backprop | loss -> chain rule -> gradients | foundation of learning |
| Token-level grads | hard tokens dominate | explains training efficiency |
| Clipping | cap gradient norm | prevents blow-ups |
| Accumulation | sum micro-batch grads | small GPU trick |
| Residual | grads can skip layers | enables deep Transformers |
| RL sparsity | fewer effective grads | RL stabilization motivation |

Gradients translate "what went wrong" into "how each parameter should change".


---

## Part Summary

1. [ok] input = full sentence without last token; labels = full sentence without first token
2. [ok] all positions predict in parallel; cross-entropy loss per position
3. [ok] total loss = mean over valid positions
4. [ok] token-level but parallel (not sentence-level, not serial tokens)
5. [ok] training uses teacher forcing; inference must generate serially
6. [ok] PAD positions are excluded via `ignore_index`

These ideas apply to all autoregressive LMs.

-> Next: generation.


### 10. From dialogue to tokens: what does JSONL actually feed the model?

A chat template formats messages into one continuous token sequence.

#### 10.1 Real training data (JSONL)

```jsonl
{"messages": [{"role": "system", "content": "You are a math assistant"}, {"role": "user", "content": "1+1=?"}, {"role": "assistant", "content": "1+1=2"}]}
{"messages": [{"role": "user", "content": "Hello"}, {"role": "assistant", "content": "Hi! How can I help?"}]}
{"messages": [{"role": "system", "content": "You are a translator"}, {"role": "user", "content": "Hello"}, {"role": "assistant", "content": "Hi"}]}
```

Core questions:
1. How messages are concatenated into one sequence
2. Which tokens are context vs targets
3. How special tokens are handled in loss


In [ ]:
# ============================================================
# 用真实的 transformers 库看 Chat Template 做了什么
# ============================================================
print("=== 真实 tokenizer demo：Chat Template 怎么把 messages 变成 token ===\n")

# 尝试加载 Qwen2.5 的 tokenizer（如果是第一次运行会自动下载）
try:
    from transformers import AutoTokenizer
    
    # Qwen2.5-0.5B 的 tokenizer 很小（~30MB），下载快
    # 它的 chat template 是 ChatML 风格，和 DeepSeek、Qwen 系一致
    MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
    
    print(f"加载 tokenizer: {MODEL_NAME}")
    print("(首次运行会下载 ~30MB，请稍候...)\n")
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    
    # 先看看 tokenizer 有哪些 special token
    print("=== Tokenizer 的 Special Token ===")
    print(f"  bos_token:        {repr(tokenizer.bos_token)} -> id={tokenizer.bos_token_id}")
    print(f"  eos_token:        {repr(tokenizer.eos_token)} -> id={tokenizer.eos_token_id}")
    print(f"  pad_token:        {repr(tokenizer.pad_token)} -> id={tokenizer.pad_token_id}")
    print(f"  词表大小:          {len(tokenizer)} 个 token")
    print()
    
    # 看看 chat template 本身（就是个 Jinja2 模板字符串！）
    print("=== Chat Template（Jinja2 模板）===")
    ct = tokenizer.chat_template
    if ct:
        # 只显示前 500 个字符
        print(ct[:500])
        print("...")
    print()
    
    # ============================================================
    # 核心demo：apply_chat_template 到底做了什么？
    # ============================================================
    messages = [
        {"role": "system", "content": "you是math assistant"},
        {"role": "user", "content": "1+1=?"},
        {"role": "assistant", "content": "1+1=2"},
    ]
    
    print("=== Input：messages 列表 ===")
    import json
    print(json.dumps(messages, ensure_ascii=False, indent=2))
    print()
    
    # Step 1: tokenize=False — 看渲染后的文本（人类可读）
    print("=== Step 1: apply_chat_template(tokenize=False) — 渲染成文本 ===")
    rendered_text = tokenizer.apply_chat_template(
        messages, 
        tokenize=False,            # 不转 token，先看文本
        add_generation_prompt=False # 不加生成提示（train时不需要）
    )
    print("渲染后的文本（注意 special token 的pos）:")
    print(repr(rendered_text))
    print()
    print("可视化:")
    print(rendered_text)
    print()
    
    # Step 2: tokenize=True — 直接得到 input_ids
    print("=== Step 2: apply_chat_template(tokenize=True) — 直接得到 token IDs ===")
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
        return_tensors="pt"  # 返回 PyTorch tensor
    )
    print(f"input_ids 形状: {input_ids.shape}")  # [batch=1, seq_len]
    print(f"input_ids 内容: {input_ids[0].tolist()}")
    print(f"序列长度: {len(input_ids[0])}")
    print()
    
    # Step 3: 每个 token 解码出来看
    print("=== Step 3: 逐个 token 解码 — 看每段是什么 ===")
    print(f"{'Position':<5s} {'Token ID':>8s} {'解码文本':<30s} {'说明'}")
    print("-" * 75)
    
    for i, tid in enumerate(input_ids[0].tolist()):
        decoded = tokenizer.decode([tid])
        # 判断 token 类型
        if tid == tokenizer.bos_token_id:
            note = "← BOS (开始标记)"
        elif tid == tokenizer.eos_token_id:
            note = "← EOS (结束标记)"
        elif tid >= len(tokenizer) - 20:  # 特殊 token 通常在词表末尾
            note = "← special token"
        elif tid == 151644:  # Qwen 的 <|im_start|>
            note = "← <|im_start|> 特殊标记"
        elif tid == 151645:  # Qwen 的 <|im_end|>
            note = "← <|im_end|> 特殊标记"
        else:
            note = ""
        print(f"{i:<5d} {tid:>8d} {decoded:<30s} {note}")
    
    print(f"\n重要观察:")
    print(f"  1. 文本被拼成了一段连续的 token 序列（不是三个独立的数组）")
    print(f"  2. system/user/assistant 之间用 <|im_start|> 和 <|im_end|> 分割")
    print(f"  3. 整段序列包括：系统提示 + 用户问题 + 助手回答")
    print(f"  4. model会一次性sees这整个序列（通过 causal attention mask）")

except ImportError:
    print("transformers 未安装。运行: pip install transformers")
    print()
    print("手动模拟 Qwen2.5 的 chat template 行为...")
    print()
    
    # 模拟 Qwen2.5 的 token IDs（真实值）
    print("Qwen2.5 Chat Template 的渲染规则（ChatML 格式）:")
    print('  <|im_start|>system')
    print('  {system_content}')
    print('  <|im_end|>')
    print('  <|im_start|>user')
    print('  {user_content}')
    print('  <|im_end|>')
    print('  <|im_start|>assistant')
    print('  {assistant_content}')
    print('  <|im_end|>')
    print()
    
    print("用I们自己实现一个简化版来demo...")
    
except Exception as e:
    print(f"加载失败: {e}")
    print("用简化版demo...")


#### 10.2 Controlled experiment: manual concatenation vs `apply_chat_template`

If both methods produce identical token IDs, then `apply_chat_template` is essentially string concatenation + tokenization.


In [ ]:
# ============================================================
# 10.3 对照实验：自己手工concatenate vs 官方 API —— 证明一模一样
# ============================================================
# 核心思想：apply_chat_template 内部没有魔法，就是按 ChatML 格式做字符串concatenate。
# I们自己手工拼一遍，和官方结果逐项对照，证明完全一致。

# 如果真实 transformers tokenizer 没加载成功，就用一个离线可运行的简化版。
if "tokenizer" not in globals():
    print("未检测到真实 tokenizer，使用 SimpleChatTokenizer 做离线demo。")

    class SimpleChatTokenizer:
        """最小可用的 ChatML tokenizer，用来证明 template = 拼字符串 + tokenize"""
        def __init__(self):
            self.special = {"<|im_start|>": 100001, "<|im_end|>": 100002}
            self.vocab = {"\n": 10}
            self.reverse = {10: "\n", 100001: "<|im_start|>", 100002: "<|im_end|>"}

        def convert_tokens_to_ids(self, token):
            return self.special.get(token, self.vocab.get(token, -1))

        def apply_chat_template(self, messages, tokenize=False):
            text = "".join(
                f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n"
                for m in messages
            )
            return self.encode(text, add_special_tokens=False) if tokenize else text

        def encode(self, text, add_special_tokens=False):
            ids = []
            i = 0
            while i < len(text):
                matched = False
                for tok, tid in self.special.items():
                    if text.startswith(tok, i):
                        ids.append(tid)
                        i += len(tok)
                        matched = True
                        break
                if matched:
                    continue
                ch = text[i]
                if ch not in self.vocab:
                    self.vocab[ch] = 1000 + len(self.vocab)
                    self.reverse[self.vocab[ch]] = ch
                ids.append(self.vocab[ch])
                i += 1
            return ids

        def decode(self, ids):
            return "".join(self.reverse[i] for i in ids)

    tokenizer = SimpleChatTokenizer()

print("=" * 70)
print("对照实验：手工concatenate vs 官方 apply_chat_template")
print("=" * 70)

# ============================================================
# 准备对话数据
# ============================================================
messages = [
    {"role": "system", "content": "you是一个乐于助人的助手。"},
    {"role": "user", "content": "1+1等于几？"},
    {"role": "assistant", "content": "1+1等于2。"},
]

print("\n原始对话：")
for i, msg in enumerate(messages):
    print(f"  [{i}] {msg['role']}: {msg['content']}")

# ============================================================
# 先看看特殊 token 的 ID（ChatML 格式的骨架）
# ============================================================
IM_START_ID = tokenizer.convert_tokens_to_ids("<|im_start|>")
IM_END_ID   = tokenizer.convert_tokens_to_ids("<|im_end|>")
NEWLINE_ID  = tokenizer.convert_tokens_to_ids("\n")

print(f"\n特殊 token ID：")
print(f"  '<|im_start|>' -> ID = {IM_START_ID}")
print(f"  '<|im_end|>'   -> ID = {IM_END_ID}")
print(f"  '\\n'           -> ID = {NEWLINE_ID}")

# ============================================================
# 方法一：官方 API —— apply_chat_template
# ============================================================
print("\n" + "=" * 70)
print("方法一：官方 API —— apply_chat_template")
print("=" * 70)

# tokenize=False -> 得到字符串（人类可读）
official_text = tokenizer.apply_chat_template(messages, tokenize=False)
# tokenize=True  -> 得到 token ID 列表
official_ids = tokenizer.apply_chat_template(messages, tokenize=True)

print(f"\n官方渲染的文本（repr）：")
print(f"  {repr(official_text)}")

print(f"\n官方渲染的文本（人类可读）：")
print(f"  {official_text}")

print(f"\n官方 token IDs（完整列表）：")
print(f"  {official_ids}")

# 逐 token 解码官方结果
print(f"\n官方结果逐 token 解码：")
print(f"  {'Position':<5s} {'ID':>7s}  {'解码':<18s}  {'标记'}")
print(f"  {'-'*55}")
for i, tid in enumerate(official_ids):
    decoded = repr(tokenizer.decode([tid]))
    marker = "<-- 特殊token" if tid in (IM_START_ID, IM_END_ID) else ""
    print(f"  [{i:>3d}] {tid:>7d}  {decoded:<18s}  {marker}")
print(f"  总 token 数：{len(official_ids)}")

# ============================================================
# 方法二：自己手工concatenate（模拟 apply_chat_template 内部逻辑）
# ============================================================
print("\n" + "=" * 70)
print("方法二：手工concatenate（模拟 ChatML 格式）")
print("=" * 70)

# ChatML 格式规则：
#   每条消息 = <|im_start|>角色\n内容<|im_end|>\n
#   所有消息按顺序首尾相连，构成一个连续字符串
#   这就是 apply_chat_template 内部 Jinja2 模板在做的事！

print("""
ChatML concatenate规则（和 Jinja2 模板做的事情一模一样）：

  消息1_str = "<|im_start|>system\\nyou是一个乐于助人的助手。<|im_end|>\\n"
  消息2_str = "<|im_start|>user\\n1+1等于几？<|im_end|>\\n"
  消息3_str = "<|im_start|>assistant\\n1+1等于2。<|im_end|>\\n"
                            |
                            v
  最终序列  = 消息1_str + 消息2_str + 消息3_str
            = 一段连续的字符串！
""")

all_text = ""    # 累积文本
all_ids = []     # 累积 token ID

print("开始逐步concatenate……\n")

for step, msg in enumerate(messages):
    role = msg["role"]
    content = msg["content"]

    # 按照 ChatML 格式构造这一段
    segment_text = f"<|im_start|>{role}\n{content}<|im_end|>\n"
    # 用 tokenizer 把这段文本编码（不加 add_special_tokens，因为这不是独立句子）
    segment_ids = tokenizer.encode(segment_text, add_special_tokens=False)

    before_len = len(all_ids)
    all_text += segment_text      # <-- 字符串concatenate！
    all_ids.extend(segment_ids)   # <-- token concatenate！

    print(f"--- 第 {step+1} 步：concatenate {role} 消息 ---")
    print(f"  这一段文本：")
    print(f"    {repr(segment_text)}")
    print(f"  这一段 token IDs ({len(segment_ids)} 个)：")
    print(f"    {segment_ids}")
    print(f"  累积 token 数：{before_len} -> {len(all_ids)}（+{len(segment_ids)}）")
    print(f"  当前累积文本 repr：")
    print(f"    {repr(all_text)}")
    print()

# ============================================================
# ★ 核心对照：官方 vs 手工 —— 逐项验证
# ============================================================
print("=" * 70)
print("对照验证：官方 API vs 手工concatenate")
print("=" * 70)

# 验证 1：文本字符串是否完全一致？
print(f"\n[验证 1] 文本字符串对比")
print(f"  官方文本 == 手工文本 : {official_text == all_text}")
if official_text != all_text:
    print(f"  官方长度 = {len(official_text)}, 手工长度 = {len(all_text)}")
    # 找出第一个不同字符
    for i, (a, b) in enumerate(zip(official_text, all_text)):
        if a != b:
            print(f"  第一个差异在pos {i}：官方={repr(a)} vs 手工={repr(b)}")
            print(f"     官方周围: ...{repr(official_text[max(0,i-10):i+10])}...")
            print(f"     手工周围: ...{repr(all_text[max(0,i-10):i+10])}...")
            break
else:
    print(f"  字符串完全一致！手工拼出来的和官方 API 一模一样！")

# 验证 2：token ID 序列是否完全一致？
print(f"\n[验证 2] token ID 序列对比")
manual_ids_list = list(all_ids)
print(f"  官方 IDs == 手工 IDs : {official_ids == manual_ids_list}")
if official_ids != manual_ids_list:
    print(f"  官方 {len(official_ids)} 个 token，手工 {len(manual_ids_list)} 个 token")
    for i in range(min(len(official_ids), len(manual_ids_list))):
        if official_ids[i] != manual_ids_list[i]:
            print(f"  第一个差异在pos {i}：")
            print(f"     官方 ID={official_ids[i]} -> {repr(tokenizer.decode([official_ids[i]]))}")
            print(f"     手工 ID={manual_ids_list[i]} -> {repr(tokenizer.decode([manual_ids_list[i]]))}")
            break
else:
    print(f"  {len(official_ids)} 个 token ID 完全一致！")

# 验证 3：两个序列反向 decode 回去，能还原成一样的文本吗？
print(f"\n[验证 3] 反向 decode 验证")
off_decoded = tokenizer.decode(official_ids)
man_decoded = tokenizer.decode(manual_ids_list)
print(f"  官方 IDs decode: {repr(off_decoded)}")
print(f"  手工 IDs decode: {repr(man_decoded)}")
print(f"  两者一致: {off_decoded == man_decoded}")

# ============================================================
# ★ 最终总结
# ============================================================
print("\n" + "=" * 70)
print("Conclusion")
print("=" * 70)
print("""
  apply_chat_template 的 Jinja2 模板，内部做的事情就是：

  1. 遍历 messages 列表
  2. 每条消息按 "<|im_start|>角色\\n内容<|im_end|>\\n" 格式化成字符串
  3. 把所有字符串按顺序首尾相连，拼成一段连续的文本
  4. tokenize 这段文本，得到一串 token ID

  没有魔法，没有隐藏操作。
  you手工做的和官方 API 做的，结果一模一样。

  理解了这个，you就知道了：
  train数据是怎么从 JSON 变成一行一行的 token 序列的。
""")


#### 10.3 The key step: building `labels`

Only assistant content contributes to loss. System/user/special tokens are ignored (set to -100).


In [ ]:
# ============================================================
# labels 构造 + ignore_index 的底层原理
# ============================================================
import torch
import torch.nn.functional as F
import math

print("=== labels 构造 + ignore_index 验证 ===\n")

# 复用前面的 vocab 和 tokenize
vocab = {
    "<|im_start|>": 151644, "<|im_end|>": 151645,
    "system": 8948, "user": 872, "assistant": 78191,
    "you": 9942, "是": 10603, "数学": 107659, "助手": 113738,
    "1": 16, "+": 17, "2": 18, "=": 19, "?": 20, "\n": 198,
}
id_to_word = {v: k for k, v in vocab.items()}

def encode(text):
    tokens = []
    i = 0
    while i < len(text):
        matched = None
        for word in sorted(vocab.keys(), key=lambda x: -len(x)):
            if text[i:].startswith(word):
                matched = word
                break
        if matched:
            tokens.append(vocab[matched])
            i += len(matched)
        else:
            tokens.append(0)
            i += 1
    return tokens

messages = [
    {"role": "system", "content": "you是math assistant"},
    {"role": "user", "content": "1+1=?"},
    {"role": "assistant", "content": "1+1=2"},
]

IM_START = "<|im_start|>"
IM_END = "<|im_end|>"
IGNORE = -100  # PyTorch 的默认 ignore_index

# ============================================================
# Step 1: 构造 input_ids
# ============================================================
print("Step 1: 构造 input_ids")
all_text = ""
for msg in messages:
    all_text += f"{IM_START}{msg['role']}\n{msg['content']}{IM_END}\n"

input_ids = torch.tensor([encode(all_text)])
print(f"  input_ids: {input_ids[0].tolist()}")
print()

# ============================================================
# Step 2: 构造 labels — 追踪每个 token 的来源
# ============================================================
print("Step 2: 构造 labels（每个 token 逐一标注）")
print()

labels = torch.full_like(input_ids, IGNORE)  # 初始全部 IGNORE

pos = 0
for msg in messages:
    role = msg["role"]
    content = msg["content"]
    
    # 头部: <|im_start|>role\n -> IGNORE（本来就是 IGNORE，可以显式确认）
    header = f"{IM_START}{role}\n"
    hlen = len(encode(header))
    # 这 hlen 个 token 已经是 IGNORE
    pos += hlen
    
    # 内容
    cids = encode(content)
    if role == "assistant":
        # ★ 只有 assistant 的内容才填入真实 label
        for cid in cids:
            labels[0, pos] = cid
            pos += 1
    else:
        # system/user 的内容保持 IGNORE
        pos += len(cids)
    
    # 尾部: <|im_end|>\n -> IGNORE
    footer = f"{IM_END}\n"
    flen = len(encode(footer))
    pos += flen

print(f"  input_ids: {input_ids[0].tolist()}")
print(f"  labels:    {labels[0].tolist()}")
print()

# 逐 token 对比
print("逐 token 对比 (L=in loss, .=ignore):")
print(f"  {'Pos':<4s} {'input_id':>8s} {'token':<18s} {'label':>8s} {'来源'}")
print(f"  {'-'*4} {'-'*8} {'-'*18} {'-'*8} {'-'*30}")

pos = 0
for msg in messages:
    role = msg["role"]
    content = msg["content"]
    
    header_ids = encode(f"{IM_START}{role}\n")
    for hid in header_ids:
        word = id_to_word.get(hid, "???")
        lid = labels[0, pos].item()
        m = "." if lid == IGNORE else "L"
        print(f"  [{pos:<2d}] {hid:>8d} {word:<18s} {lid:>8d} {m} 框架({role})")
        pos += 1
    
    content_ids = encode(content)
    for cid in content_ids:
        word = id_to_word.get(cid, "???")
        lid = labels[0, pos].item()
        m = "L" if lid != IGNORE else "."
        note = f"{m} [{role}内容]"
        print(f"  [{pos:<2d}] {cid:>8d} {word:<18s} {lid:>8d} {note}")
        pos += 1
    
    footer_ids = encode(f"{IM_END}\n")
    for fid in footer_ids:
        word = id_to_word.get(fid, "???")
        lid = labels[0, pos].item()
        m = "." if lid == IGNORE else "L"
        print(f"  [{pos:<2d}] {fid:>8d} {word:<18s} {lid:>8d} {m} 框架标记")
        pos += 1

n_compute = (labels != IGNORE).sum().item()
n_ignore = (labels == IGNORE).sum().item()
print(f"\n  统计: in loss 的 token={n_compute} 个, ignore的 token={n_ignore} 个")
print(f"  有效train信号占比: {n_compute}/{n_compute+n_ignore} = {n_compute/(n_compute+n_ignore)*100:.1f}%")

# ============================================================
# Step 3: 验证 ignore_index 真的跳过了 -100 的pos
# ============================================================
print(f"\n{'='*60}")
print("Step 3: 验证 ignore_index 机制")
print("=" * 60)

# 模拟model输出 logits（随机）
VOCAB_SIZE = max(vocab.values()) + 1
torch.manual_seed(42)
logits = torch.randn(1, input_ids.shape[1], VOCAB_SIZE)

# 方式 A: 使用 ignore_index（正常做法）
loss_with_ignore = F.cross_entropy(
    logits.reshape(-1, VOCAB_SIZE),
    labels.reshape(-1),
    ignore_index=IGNORE
)
print(f"\n使用 ignore_index={IGNORE} 的 loss: {loss_with_ignore.item():.4f}")

# 方式 B: 手工验证 — 只算 labels != IGNORE 的pos
losses = F.cross_entropy(
    logits.reshape(-1, VOCAB_SIZE),
    labels.reshape(-1),
    ignore_index=IGNORE,
    reduction='none'  # 不做平均，保留每个pos的 loss
)
losses_reshaped = losses.view(input_ids.shape)  # [1, seq_len]

print(f"\n每个pos的 loss (ignore_index 已自动把 -100 pos归零):")
for i in range(input_ids.shape[1]):
    tid = input_ids[0, i].item()
    lid = labels[0, i].item()
    pos_loss = losses_reshaped[0, i].item()
    word = id_to_word.get(tid, "???")
    if lid == IGNORE:
        print(f"  [{i:2d}] loss={pos_loss:.4f} ← IGNORE pos, loss=0")
    else:
        print(f"  [{i:2d}] loss={pos_loss:.4f} ← 有效pos! predict'{word}'")

# 手工算平均
manual_avg = losses_reshaped[labels != IGNORE].mean()
print(f"\n手工平均（只算有效pos）: {manual_avg:.4f}")
print(f"PyTorch cross_entropy 结果: {loss_with_ignore.item():.4f}")
print(f"一致 [ok]" if abs(manual_avg.item() - loss_with_ignore.item()) < 1e-5 else "不一致 [x]")

# ============================================================
# Step 4: 如果不用 ignore_index 会怎样？（对比）
# ============================================================
print(f"\n{'='*60}")
print("Step 4: 对比 — 如果不用 ignore_index")
print("=" * 60)

# 把 labels 里的 -100 替换为一个真实 token ID（比如 0=UNK）
labels_bad = labels.clone()
labels_bad[labels_bad == IGNORE] = 0

loss_bad = F.cross_entropy(
    logits.reshape(-1, VOCAB_SIZE),
    labels_bad.reshape(-1)
)
print(f"把所有 -100 改成 0 后的 loss: {loss_bad.item():.4f}")
print(f"正确的 loss (ignore -100):     {loss_with_ignore.item():.4f}")
print(f"差异: {abs(loss_bad.item() - loss_with_ignore.item()):.4f}")
print()
print("-> 如果不用 ignore_index，model会被迫学习:")
print("  '在系统提示后面必须predict<|im_end|>'")
print("  '在用户问题后面必须predict<|im_end|>'")
print("  这些都是无意义的噪声！")
print()
print("总结:")
print("  1. labels 中 assistant 内容 = 真实 token ID -> in loss -> model学到生成答案")
print("  2. labels 中其他一切 = -100 (ignore_index) -> 不in loss -> model只看不学")
print("  3. CrossEntropyLoss(ignore_index=-100) 自动跳过，loss=0, grad=0")

#### 10.4 Multi-turn dialogues

Concatenation rule is the same: append turns in order.
Labels rule is the same: only assistant content contributes to loss.


In [ ]:
# ============================================================
# multi-turn dialogueconcatenate证明：5 条消息也是拼成一段
# ============================================================
import torch
import torch.nn.functional as F

print("=" * 70)
print("证明：multi-turn dialogue也是concatenate成一段连续 token 序列")
print("=" * 70)
print()

# 词表
vocab = {
    "<|im_start|>": 151644, "<|im_end|>": 151645,
    "system": 8948, "user": 872, "assistant": 78191,
    "you": 9942, "是": 10603, "数学": 107659, "老师": 113740,
    "1": 16, "+": 17, "2": 18, "=": 19, "?": 20, "。": 21,
    "3": 22, "4": 23, "那": 104322, "呢": 104535, "\n": 198,
}
id_to_word = {v: k for k, v in vocab.items()}

def encode(text):
    tokens = []
    i = 0
    while i < len(text):
        matched = None
        for word in sorted(vocab.keys(), key=lambda x: -len(x)):
            if text[i:].startswith(word):
                matched = word
                break
        if matched:
            tokens.append(vocab[matched])
            i += len(matched)
        else:
            tokens.append(0)
            i += 1
    return tokens

IM_START = "<|im_start|>"
IM_END = "<|im_end|>"
IGNORE = -100

# multi-turn dialogue数据（2 轮）
multi_turn = {
    "messages": [
        {"role": "system", "content": "you是math teacher"},
        {"role": "user", "content": "1+1=?"},
        {"role": "assistant", "content": "1+1=2。"},
        {"role": "user", "content": "那2+2呢?"},
        {"role": "assistant", "content": "2+2=4。"},
    ]
}

print("multi-turn dialogue数据（5 条消息 = system + 2轮×2）:")
for i, msg in enumerate(multi_turn["messages"]):
    print(f"  [{i}] {msg['role']:>10s}: {msg['content']}")

# ============================================================
# 逐步concatenate — 5 条消息
# ============================================================
print()
print("=" * 70)
print("逐步concatenate — 5 条消息全部拼成一段")
print("=" * 70)

all_text = ""
all_ids = []

for step, msg in enumerate(multi_turn["messages"]):
    role = msg["role"]
    content = msg["content"]
    
    segment = f"{IM_START}{role}\n{content}{IM_END}\n"
    segment_ids = encode(segment)
    
    before_len = len(all_ids)
    all_text += segment
    all_ids.extend(segment_ids)
    
    print(f"\nStep {step+1}/5: concatenate {role} -> \"{content}\"")
    print(f"  片段: {repr(segment)}")
    print(f"  片段 token 数: {len(segment_ids)}")
    print(f"  累积 token 数: {before_len} -> {len(all_ids)}")
    print(f"  累积文本: {repr(all_text)}")

# ============================================================
# 最终证明
# ============================================================
print(f"\n{'='*70}")
print("[ok] 最终：5 条消息拼成了 1 段连续的 token 序列")
print(f"{'='*70}")
print(f"  总 token 数: {len(all_ids)}")
print(f"  完整 IDs: {all_ids}")
print()

# 逐 token 标注轮次
print("逐 token 标注（证明是连续序列，非分段数组）:")
print(f"{'Pos':<4s} {'ID':>7s} {'token':<16s} {'轮次/角色':<25s} {'in loss?'}")
print(f"{'-'*4} {'-'*7} {'-'*16} {'-'*25} {'-'*8}")

pos = 0
for turn_idx, msg in enumerate(multi_turn["messages"]):
    role = msg["role"]
    content = msg["content"]
    
    header_ids = encode(f"{IM_START}{role}\n")
    for hid in header_ids:
        word = id_to_word.get(hid, "???")
        print(f"{pos:<4d} {hid:>7d} {word:<16s} {'第'+str(turn_idx+1)+'轮 框架('+role+')':<25s} ignore")
        pos += 1
    
    content_ids = encode(content)
    for cid in content_ids:
        word = id_to_word.get(cid, "???")
        if role == "assistant":
            loss_note = "✓ 计算!"
        else:
            loss_note = "ignore"
        print(f"{pos:<4d} {cid:>7d} {word:<16s} {'第'+str(turn_idx+1)+'轮 '+role+'内容':<25s} {loss_note}")
        pos += 1
    
    footer_ids = encode(f"{IM_END}\n")
    for fid in footer_ids:
        word = id_to_word.get(fid, "???")
        print(f"{pos:<4d} {fid:>7d} {word:<16s} {'第'+str(turn_idx+1)+'轮 框架结尾':<25s} ignore")
        pos += 1
    
    if turn_idx < len(multi_turn["messages"]) - 1:
        print(f"{'':>4s} {'↓ 继续concatenate，序列不中断 ↓':>50s}")

print()
print(f"Conclusion:")
print(f"  5 条独立的消息 -> 1 段 {pos} 个 token 的连续序列")
print(f"  model一次性读取整段序列（通过 causal attention）")
print(f"  只有 assistant 部分的 token 参与 loss 计算")
print(f"  多轮 = 单轮的多次重复concatenate，规则完全一样")

#### 10.5 Full loop: chat template + training loop

Pipeline:
1. read JSONL
2. apply chat template -> input_ids + labels
3. shift input/labels
4. forward + CrossEntropyLoss(ignore_index=-100)
5. backward + clip + optimizer.step

Then at inference time, use the same chat template to format the prompt before generation.


In [ ]:
# ============================================================
# 完整train循环：Chat Template + MiniGPT
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

print("=== 完整train循环：对话数据 -> token -> train ===\n")

# -------- 复用之前的词表和函数 --------
vocab = {
    "<|im_start|>": 151644, "<|im_end|>": 151645,
    "system": 8948, "user": 872, "assistant": 78191,
    "you": 9942, "是": 10603, "数学": 107659, "老师": 113740,
    "1": 16, "+": 17, "2": 18, "=": 19, "?": 20, "。": 21,
    "3": 22, "4": 23, "那": 104322, "呢": 104535, "\n": 198,
    "翻译": 112345, "官": 105678,
    "Hello": 201, "you好": 202, "!": 203,
    "天气": 301, "怎么": 302, "样": 303,
    "今天": 304, "不": 305, "错": 306, "晴天": 307,
}
id_to_word = {v: k for k, v in vocab.items()}
VOCAB_SIZE = max(vocab.values()) + 10
IM_START = "<|im_start|>"
IM_END = "<|im_end|>"
IGNORE = -100
PAD_ID = 0

def encode(text):
    tokens = []
    i = 0
    while i < len(text):
        matched = None
        for word in sorted(vocab.keys(), key=lambda x: -len(x)):
            if text[i:].startswith(word):
                matched = word
                break
        if matched:
            tokens.append(vocab[matched])
            i += len(matched)
        else:
            tokens.append(0)
            i += 1
    return tokens

# -------- 准备train数据（3 条对话） --------
train_conversations = [
    {
        "messages": [
            {"role": "system", "content": "you是math teacher"},
            {"role": "user", "content": "1+1=?"},
            {"role": "assistant", "content": "1+1=2。"},
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "you是翻译官"},
            {"role": "user", "content": "Hello!"},
            {"role": "assistant", "content": "you好!"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "天气怎么样?"},
            {"role": "assistant", "content": "今天天气不错，晴天。"},
        ]
    },
]

# -------- 构造 input_ids 和 labels --------
print("=== 构造train数据 ===")
all_inputs = []
all_labels = []

for idx, conv in enumerate(train_conversations):
    messages = conv["messages"]
    
    # concatenate文本
    text = ""
    for msg in messages:
        text += f"{IM_START}{msg['role']}\n{msg['content']}{IM_END}\n"
    
    input_ids = encode(text)
    labels = [IGNORE] * len(input_ids)
    
    # 标注 assistant 内容
    pos = 0
    for msg in messages:
        role = msg["role"]
        content = msg["content"]
        pos += len(encode(f"{IM_START}{role}\n"))
        cids = encode(content)
        if role == "assistant":
            for j, cid in enumerate(cids):
                labels[pos + j] = cid
        pos += len(cids)
        pos += len(encode(f"{IM_END}\n"))
    
    all_inputs.append(input_ids)
    all_labels.append(labels)
    
    n_assistant = sum(1 for l in labels if l != IGNORE)
    n_total = len(labels)
    print(f"对话 {idx+1}: {n_total} tokens, {n_assistant} in loss ({n_assistant/n_total*100:.0f}%)")
    # 如果没有 system prompt，标注
    has_system = any(m["role"] == "system" for m in messages)
    if not has_system:
        print(f"  (没有 system prompt)")

print()

# -------- Padding 到相同长度 --------
max_len = max(len(ids) for ids in all_inputs)
print(f"最大序列长度: {max_len}")

padded_inputs = []
padded_labels = []

for input_ids, labels in zip(all_inputs, all_labels):
    pad_len = max_len - len(input_ids)
    padded_inputs.append(input_ids + [PAD_ID] * pad_len)
    padded_labels.append(labels + [IGNORE] * pad_len)

input_ids_batch = torch.tensor(padded_inputs)
labels_batch = torch.tensor(padded_labels)

print(f"input_ids_batch 形状: {input_ids_batch.shape}")
print(f"labels_batch 形状: {labels_batch.shape}")
print()
print("input_ids_batch:")
print(input_ids_batch)
print()
print("labels_batch (-100=ignore):")
print(labels_batch)
print()

# -------- 拆成 input/target --------
# 和 Part 5 开头一样的逻辑：input = 去掉最后一个, target = 去掉第一个
model_inputs = input_ids_batch[:, :-1]   # [batch, seq-1]
model_targets = labels_batch[:, 1:]       # [batch, seq-1]

print(f"modelInput形状: {model_inputs.shape}")
print(f"model目标形状: {model_targets.shape}")
print()

# -------- 创建简单model --------
class TinyLLM(nn.Module):
    def __init__(self, vocab_size, d_model=32):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model, nhead=4, dim_feedforward=64, batch_first=True),
            num_layers=2
        )
        self.lm_head = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        mask = nn.Transformer.generate_square_subsequent_mask(x.shape[1], device=x.device)
        x = self.embed(x)
        x = self.transformer(x, mask=mask, is_causal=True)
        return self.lm_head(x)

model = TinyLLM(VOCAB_SIZE, d_model=32)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
print(f"model参数量: {sum(p.numel() for p in model.parameters()):,}")
print()

# -------- train循环 --------
NUM_EPOCHS = 10
losses = []

print(f"=== train {NUM_EPOCHS} 个 epoch ===")
model.train()
for epoch in range(NUM_EPOCHS):
    logits = model(model_inputs)  # [batch=3, seq-1, vocab_size]
    
    loss = F.cross_entropy(
        logits.reshape(-1, VOCAB_SIZE),
        model_targets.reshape(-1),
        ignore_index=IGNORE  # ← Key！ignore -100 的pos
    )
    
    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    
    losses.append(loss.item())
    if (epoch + 1) % 2 == 0:
        print(f"  Epoch {epoch+1:2d}/{NUM_EPOCHS} | Loss: {loss.item():.4f}")

print(f"\nLoss: {losses[0]:.4f} -> {losses[-1]:.4f} (下降 = model在学习 assistant 回复的模式)")
print()

# -------- 推理demo：用train好的model生成 --------
print("=== 推理demo ===")
# 构造 prompt: system + user
prompt_messages = [
    {"role": "system", "content": "you是math teacher"},
    {"role": "user", "content": "2+2=?"}
]

# concatenate prompt（和train时一样的 chat template，但是结尾不同）
prompt_text = ""
for msg in prompt_messages:
    prompt_text += f"{IM_START}{msg['role']}\n{msg['content']}{IM_END}\n"
prompt_text += f"{IM_START}assistant\n"  # ← add_generation_prompt

prompt_ids = torch.tensor([encode(prompt_text)])
print(f"Prompt: {prompt_text.strip()}")
print(f"Prompt IDs: {prompt_ids[0].tolist()}")

# 自回归生成
model.eval()
generated = prompt_ids.clone()
with torch.no_grad():
    for _ in range(10):  # 最多生成 10 个 token
        logits = model(generated)
        next_logits = logits[0, -1, :]  # 最后一个pos的predict
        probs = F.softmax(next_logits / 0.7, dim=-1)
        
        # 禁止 PAD 和 special token
        probs[0] = 0
        probs[151644] = 0
        probs[151645] = 0
        
        next_token = torch.multinomial(probs, 1)
        generated = torch.cat([generated, next_token.unsqueeze(0)], dim=1)
        
        if next_token.item() == 151645:  # <|im_end|>
            break

# 解码
output_ids = generated[0].tolist()
output_text = ""
for tid in output_ids[len(prompt_ids[0]):]:  # 只取新生成的
    word = id_to_word.get(tid, f"[{tid}]")
    output_text += word
    if tid == 151645:
        break

print(f"生成文本: {output_text}")
print()

# -------- 总结 --------
print("=" * 60)
print("完整链路回顾:")
print("=" * 60)
print("""
  1. JSONL 数据 -> messages 列表
  2. Chat Template -> concatenate成带特殊 token 的文本
  3. Tokenize -> input_ids (modelsees的所有 token)
  4. 构造 labels -> 只有 assistant 内容保留真实 ID，其他全 -100
  5. train -> CrossEntropyLoss(ignore_index=-100) 只优化 assistant 部分
  6. 推理 -> 同样的 chat template concatenate prompt, add_generation_prompt=True
  7. 生成 -> 自回归（07-generation.ipynb 的内容）
  
  这就是从「对话数据」到「train好的聊天model」的完整流程。
""")